# 02 · Terrain & Hydrological Analysis

**Step 2** of the pipeline — the *Terrain Processing Engine* and *Hydrological Feature Extraction*.

We derive:
- **Terrain:** slope, aspect, roughness (TRI), curvature, hillshade.
- **Hydrology:** depression-filled DEM, D8 flow direction, flow accumulation, Topographic Wetness Index (TWI), drainage density.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, matplotlib.pyplot as plt
from src import terrain_analysis, hydrology, utils
from src.utils import STUDY_AREA, PROCESSED_DIR

dem = np.load(PROCESSED_DIR / 'dem.npy')
extent = [STUDY_AREA.min_lon, STUDY_AREA.max_lon, STUDY_AREA.min_lat, STUDY_AREA.max_lat]
print('Loaded DEM', dem.shape)

## Terrain derivatives

In [ ]:
terrain = terrain_analysis.analyze_terrain(dem)
fig, ax = plt.subplots(2, 2, figsize=(13, 11))
for a, key, cmap in zip(ax.ravel(), ['slope','aspect','roughness','hillshade'],
                        ['YlOrBr','twilight','magma','gray']):
    im = a.imshow(terrain[key], cmap=cmap, extent=extent)
    a.set_title(key.title()); fig.colorbar(im, ax=a, shrink=0.8)
plt.tight_layout(); plt.show()

## Hydrological extraction (D8)

In [ ]:
hydro = hydrology.analyze_hydrology(dem, terrain['slope'])
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
im0 = ax[0].imshow(np.log1p(hydro['flow_accumulation']), cmap='Blues', extent=extent)
ax[0].set_title('Flow accumulation (log)'); fig.colorbar(im0, ax=ax[0], shrink=0.8)
im1 = ax[1].imshow(hydro['twi'], cmap='YlGnBu', extent=extent)
ax[1].set_title('Topographic Wetness Index'); fig.colorbar(im1, ax=ax[1], shrink=0.8)
im2 = ax[2].imshow(hydro['drainage_density'], cmap='PuBu', extent=extent)
ax[2].set_title('Drainage density'); fig.colorbar(im2, ax=ax[2], shrink=0.8)
plt.tight_layout(); plt.show()

In [ ]:
for name, arr in {**terrain, **hydro}.items():
    if isinstance(arr, np.ndarray) and arr.dtype != object:
        np.save(PROCESSED_DIR / f'layer_{name}.npy', arr.astype('float32'))
print('Saved terrain + hydrology layers. Next: 03_feature_engineering.ipynb')